<a href="https://colab.research.google.com/github/weagan/MoE/blob/main/moe2_lora_walkthrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MoE & MoE²-LoRA Numerical Walkthrough

This notebook implements the step-by-step numerical examples for:
1. A basic **Top-2 Mixture-of-Experts (MoE)** layer.
2. **MoE²-LoRA's Routing-Conditioned Projection (RCP)** operating in parallel with the frozen base MoE layer.

## Part 1: Basic Top-2 Mixture-of-Experts (MoE) Layer



### Step-by-Step Walkthrough: A Token Passing Through a Top-2 MoE Layer

This example uses a total of $N = 3$ experts, with $K = 2$ active experts, and a hidden dimension of $D = 2$.

#### Setup: The Inputs

Suppose our token's current hidden state vector is:


$$x = \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix}$$

And our router matrix $W_{g}$ (which has one row for each of the 3 experts) is:


$$W_{g} = \begin{bmatrix} 1.0 & 1.0 \\  -1.0 & 1.5 \\  0.2 & 0.1 \end{bmatrix}$$

---

#### Step 1: Compute Router Logits ($z = W_g x$)

We multiply the router matrix by the token vector to get a raw score for each expert:

* **Expert 1:** $(1.0 \times 1.0) + (1.0 \times 2.0) = \mathbf{3.0}$
* **Expert 2:** $(-1.0 \times 1.0) + (1.5 \times 2.0) = \mathbf{2.0}$
* **Expert 3:** $(0.2 \times 1.0) + (0.1 \times 2.0) = \mathbf{0.4}$

So our raw logits vector is:


$$z = \begin{bmatrix} 3.0, & 2.0, & 0.4 \end{bmatrix}$$

---

#### Step 2: Top-$K$ Selection ($K = 2$)

We pick the 2 experts with the highest scores:

* **Selected:** Expert 1 ($3.0$) and Expert 2 ($2.0$).
* **Ignored:** Expert 3 ($0.4$) is dropped completely and will not run.

---

#### Step 3: Compute Gating Weights (Softmax)

We apply softmax only to the two winning scores ($3.0$ and $2.0$) so their weights sum to $1.0$:

* $e^{3.0} \approx 20.086$
* $e^{2.0} \approx 7.389$
* $\text{Sum} = 20.086 + 7.389 = 27.475$

Now divide each by the sum to get the gating weights ($g$):

* **Weight for Expert 1 ($g_{1}$):** $\frac{20.086}{27.475} \approx \mathbf{0.731}$ (73.1%)
* **Weight for Expert 2 ($g_{2}$):** $\frac{7.389}{27.475} \approx \mathbf{0.269}$ (26.9%)

---

#### Step 4: Run Active Experts & Blend Outputs

Now the token $x = \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix}$ is fed through Expert 1 and Expert 2. Suppose their neural networks produce these output vectors:

* $E_1(x) = \begin{bmatrix} 3.0 \\ -1.0 \end{bmatrix}$
* $E_2(x) = \begin{bmatrix} 0.0 \\ 4.0 \end{bmatrix}$

We multiply each output by its gating weight and add them together:


$$\text{MoE}(x) = 0.731 \begin{bmatrix} 3.0 \\ -1.0 \end{bmatrix} + 0.269 \begin{bmatrix} 0.0 \\ 4.0 \end{bmatrix}$$

$$\text{MoE}(x) = \begin{bmatrix} 2.193 \\ -0.731 \end{bmatrix} + \begin{bmatrix} 0.000 \\ 1.076 \end{bmatrix} = \begin{bmatrix} 2.193 \\ 0.345 \end{bmatrix}$$

---

#### Step 5: Add to the Residual Stream

Finally, this blended MoE result is added back onto the original input token $x$ before moving to the next layer of the Transformer:


$$x_{\text{next}} = x + \text{MoE}(x) = \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix} + \begin{bmatrix} 2.193 \\ 0.345 \end{bmatrix} = \begin{bmatrix} \mathbf{3.193} \\ \mathbf{2.345} \end{bmatrix}$$

---

### How MoE²-LoRA Plugs Into This Exact Example

In MoE²-LoRA, those exact raw logits from Step 1 ($z = [3.0, 2.0, 0.4]$) get concatenated with a compressed version of $x$ to pick two small LoRA "cheat sheets" from the shared pool. Those LoRA outputs are simply added onto $x_{\text{next}}$ in Step 5 alongside $\text{MoE}(x)$.

---

### Compact One-Line Tensor Chain

$$
\underbrace{x}_{[B, S, D]}
\xrightarrow{\text{Flatten}} \underbrace{x_{\text{flat}}}_{[T, D]}
\xrightarrow{\text{Router } W_g} \underbrace{z}_{[T, N]}
\xrightarrow{\text{Top-K}} \underbrace{z_{\text{topK}}}_{[T, K]}
\xrightarrow{\text{Softmax}} \underbrace{g}_{[T, K]}
\xrightarrow{\text{Dispatch \& } E_i} \underbrace{E_i(x_{\text{flat}})}_{[T_i, D]}
\xrightarrow{\sum g_i E_i} \underbrace{\text{MoE}(x)}_{[T, D]}
\xrightarrow{\text{Reshape } + x} \underbrace{y}_{[B, S, D]}
$$


### Quick Dimension Key

For easy reference while working through the architecture or writing code, here is the quick dimension key for tensor shapes:

* $[B, S, D]$: Batch size ($B$), Sequence length ($S$), Hidden dimension ($D$).
* $[T, D]$: Flattened tokens ($T = B \times S$), Hidden dimension ($D$).
* $[T, N]$: Raw scores for all $N$ experts per token.
* $[T, K]$: Selected top $K$ expert scores/weights per token.
* $[T_i, D]$: Subset of tokens routed to expert $i$.

### Walkthrough of the Compact One-Line Tensor Chain

#### 1. Input to Flattened Tokens
$$\underbrace{x}_{[B, S, D]} \xrightarrow{\text{Flatten}} \underbrace{x_{\text{flat}}}_{[T, D]}$$
* **What happens:** The input enters as a 3D tensor containing $B$ sequences of length $S$, where each token is a vector of size $D$.
* **Why:** An MoE router makes decisions for individual tokens rather than whole sentences. Merging the batch and sequence dimensions creates a 2D matrix of $T = B \times S$ independent tokens.

#### 2. Scoring All Experts (Routing)
$$\underbrace{x_{\text{flat}}}_{[T, D]} \xrightarrow{\text{Router } W_g} \underbrace{z}_{[T, N]}$$
* **What happens:** Every token vector is multiplied by the router weight matrix $W_g$.
* **Why:** This assigns $N$ raw preference scores (logits) to every single token, measuring how well each of the $N$ experts fits that token.

#### 3. Picking the Top-$K$ Winners
$$\underbrace{z}_{[T, N]} \xrightarrow{\text{Top-K}} \underbrace{z_{\text{topK}}}_{[T, K]}$$
* **What happens:** For each token row, the model keeps only the $K$ highest scores and discards the remaining $N - K$ scores.
* **Why:** This creates **sparsity**. Instead of running all $N$ neural networks, the model only activates a tiny fraction ($K$) per token to save compute.

#### 4. Normalizing Weights (Softmax)
$$\underbrace{z_{\text{topK}}}_{[T, K]} \xrightarrow{\text{Softmax}} \underbrace{g}_{[T, K]}$$
* **What happens:** The $K$ winning scores are passed through a softmax function.
* **Why:** Softmax converts raw scores into positive percentages (gating weights $g$) that add up to $1.0$, determining how much influence each chosen expert will have on the final answer.

#### 5. Dispatching Tokens & Running Active Experts
$$\underbrace{g}_{[T, K]} \xrightarrow{\text{Dispatch \& } E_i} \underbrace{E_i(x_{\text{flat}})}_{[T_i, D]}$$
* **What happens:** Each token is routed only to its $K$ assigned experts. Expert $i$ processes its specific subset of $T_i$ tokens through its feed-forward network $E_i(\cdot)$.
* **Why:** Because different tokens go to different experts, each expert only processes a slice of the batch ($T_i \leq T$), returning transformed vectors of the same hidden size $D$.

#### 6. Weighted Blending
$$\underbrace{E_i(x_{\text{flat}})}_{[T_i, D]} \xrightarrow{\sum g_i E_i} \underbrace{\text{MoE}(x)}_{[T, D]}$$
* **What happens:** Each expert's output vector is multiplied by its gating percentage $g_i$ from Step 4, and the $K$ results for each token are added together.
* **Why:** This recombines the specialized work of the active experts back into a single unified vector per token.

#### 7. Restoring Shape & Residual Addition
$$\underbrace{\text{MoE}(x)}_{[T, D]} \xrightarrow{\text{Reshape } + x} \underbrace{y}_{[B, S, D]}$$
* **What happens:** The 2D token matrix $[T, D]$ is reshaped back into its original 3D sequence format $[B, S, D]$, and the original input $x$ is added back in via a residual connection.
* **Why:** This preserves the model's existing information stream and formats the tensor so the next Transformer layer can read it.

---

### Quick Dimension Key
* **$[B, S, D]$**: Batch size ($B$), Sequence length ($S$), Hidden dimension ($D$).
* **$[T, D]$**: Flattened tokens ($T = B \times S$), Hidden dimension ($D$).
* **$[T, N]$**: Raw scores for all $N$ experts per token.
* **$[T, K]$**: Selected top $K$ expert scores/weights per token.
* **$[T_i, D]$**: Subset of $T_i$ tokens routed to expert $i$.


### Backpropagation Through a Basic Top-$K$ MoE Layer

Given the forward pass $y = \sum_{i \in \mathcal{T}_K} g_i(x) E_i(x)$ and upstream gradient $\delta_y = \frac{\partial \mathcal{L}}{\partial y} \in \mathbb{R}^D$:

#### 1. Active Expert Gradients ($\theta_{E_i}$)
$$\frac{\partial \mathcal{L}}{\partial \theta_{E_i}} = \begin{cases} g_i(x) \left( \dfrac{\partial E_i(x)}{\partial \theta_{E_i}} \right)^\top \delta_y & \text{if } i \in \mathcal{T}_K \\ 0 & \text{if } i \notin \mathcal{T}_K \end{cases}$$

#### 2. Router Logit Gradients ($z_i$) via Softmax Jacobian
First compute the scalar alignment gradient for each active gate:
$$\frac{\partial \mathcal{L}}{\partial g_i} = \delta_y^\top E_i(x) \quad (i \in \mathcal{T}_K)$$

Then backpropagate through the Top-$K$ softmax:
$$\frac{\partial \mathcal{L}}{\partial z_i} = \begin{cases} g_i(x) \left( \dfrac{\partial \mathcal{L}}{\partial g_i} - \sum_{j \in \mathcal{T}_K} g_j(x) \dfrac{\partial \mathcal{L}}{\partial g_j} \right) & \text{if } i \in \mathcal{T}_K \\ 0 & \text{if } i \notin \mathcal{T}_K \end{cases}$$

#### 3. Router Matrix ($W_g$) & Input ($x$) Gradients
$$\frac{\partial \mathcal{L}_{\text{total}}}{\partial W_g} = \left( \frac{\partial \mathcal{L}}{\partial z} \right) x^\top + \frac{\partial \mathcal{L}_{\text{aux}}}{\partial W_g}$$

$$\frac{\partial \mathcal{L}}{\partial x} = \sum_{i \in \mathcal{T}_K} g_i(x) \left( \frac{\partial E_i(x)}{\partial x} \right)^\top \delta_y + W_g^\top \left( \frac{\partial \mathcal{L}}{\partial z} \right)$$


In [ ]:
import numpy as np

x = np.array([1.0, 2.0])
W_g = np.array([
    [1.0, 1.0],
    [-1.0, 1.5],
    [0.2, 0.1]
])
K = 2

# Step 1. Compute Router Logits (z = W_g x)
z = W_g @ x
print("Step 1 - Router Logits (z):", np.round(z, 4))

# Step 2. Top-K Selection (K = 2)
top_k_indices = np.argsort(z)[::-1][:K]
top_k_logits = z[top_k_indices]
print("Step 2 - Selected Expert Indices (0-based):", top_k_indices)
print("Step 2 - Selected Expert Logits:", np.round(top_k_logits, 4))

# Step 3. Compute Gating Weights (Softmax over Top-K)
exp_logits = np.exp(top_k_logits)
g = exp_logits / np.sum(exp_logits)
print("Step 3 - Gating Weights (g):", np.round(g, 4))

# Step 4. Run Active Experts & Blend Outputs
E1_x = np.array([3.0, -1.0])
E2_x = np.array([0.0, 4.0])
moe_out = g[0] * E1_x + g[1] * E2_x
print("Step 4 - Blended MoE Output:", np.round(moe_out, 4))

# Step 5. Add to the Residual Stream
x_next_base = x + moe_out
print("Step 5 - Next Hidden State (Base MoE):", np.round(x_next_base, 4))


## Part 2: MoE²-LoRA Routing-Conditioned Projection (RCP)

### Routing-Conditioned Projection (RCP): Step-by-Step Walkthrough

Building directly on the previous example, Routing-Conditioned Projection (RCP) runs in parallel with the frozen base experts. It takes the exact token vector and raw router logits we already computed and uses them to select from a global pool of $N_L = 3$ LoRA experts (picking the top $K_L = 2$).

---

#### Recap: What We Already Have From the Base Layer

* **Input Token ($x$):**

$$x = \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix} \quad (\text{hidden dimension } D = 2)$$


* **Frozen Base Router Logits ($z$):**

$$z = \begin{bmatrix} 3.0, & 2.0, & 0.4 \end{bmatrix} \quad (\text{3 base experts})$$


* **Base MoE Output ($\text{MoE}(x)$):**

$$\text{MoE}(x) = \begin{bmatrix} 2.193 \\ 0.345 \end{bmatrix}$$



---

#### Step 1: Compress the Token ($h_c = W_h x$)

To inject a task-specific signal without doing expensive math in the full hidden dimension $D$, RCP projects $x$ down into a tiny bottleneck dimension $d_b = 1$ using a small learnable matrix $W_h \in \mathbb{R}^{1 \times 2}$:

$$W_{h} = \begin{bmatrix} 0.5 & -0.2 \end{bmatrix}$$

$$h_{c} = \begin{bmatrix} 0.5 & -0.2 \end{bmatrix} \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix} = (0.5 \times 1.0) + (-0.2 \times 2.0) = \mathbf{0.1}$$

---

#### Step 2: Concatenate Both Channels ($[z ; h_c]$)

Next, RCP glues the frozen base router's raw logits ($z$) together with the compressed token signal ($h_{c}$). This gives the LoRA router a combined view of what the pretrained model thinks plus what the new fine-tuning task needs:

$$[z;h_{c}] = \begin{bmatrix} 3.0, & 2.0, & 0.4, & 0.1 \end{bmatrix}^{T}$$

---

#### Step 3: Score the Global LoRA Pool ($p_L = \text{softmax}(W_l \cdot [z ; h_c])$)

This layer's learnable LoRA router matrix $W_l \in \mathbb{R}^{3 \times 4}$ maps that 4-element combined vector into raw scores for the $N_L = 3$ LoRA adapters in the shared library:

$$W_{l} = \begin{bmatrix} 0.5 & 0.1 & -0.5 & 1.0 \\  -0.2 & 0.8 & 0.0 & 0.5 \\  0.1 & -0.5 & 1.0 & -1.0 \end{bmatrix}$$

Multiplying $W_l \cdot [z ; h_c]$ gives raw LoRA logits:

* **LoRA 1:** $(0.5)(3.0) + (0.1)(2.0) + (-0.5)(0.4) + (1.0)(0.1) = \mathbf{1.60}$
* **LoRA 2:** $(-0.2)(3.0) + (0.8)(2.0) + (0.0)(0.4) + (0.5)(0.1) = \mathbf{1.05}$
* **LoRA 3:** $(0.1)(3.0) + (-0.5)(2.0) + (1.0)(0.4) + (-1.0)(0.1) = \mathbf{-0.40}$

Applying softmax across all 3 scores ($e^{1.60} \approx 4.953$, $e^{1.05} \approx 2.858$, $e^{-0.40} \approx 0.670$, $\text{sum} = 8.481$) yields the full probability distribution $p_{L}$:

$$p_{L} = \begin{bmatrix} \mathbf{0.5840}, & \mathbf{0.3369}, & \mathbf{0.0790} \end{bmatrix}$$

---

#### Step 4: Top-$K_{L}$ Selection & Renormalization ($K_L = 2$)

We pick the top 2 LoRA adapters (LoRA 1 and LoRA 2) and renormalize their probabilities so they sum to $1.0$:

* **Sum of Top 2:** $0.5840 + 0.3369 = 0.9209$
* **Weight for LoRA 1 ($\tilde{p}_{L,1}$):** $\frac{0.5840}{0.9209} \approx \mathbf{0.6341}$ (63.4%)
* **Weight for LoRA 2 ($\tilde{p}_{L,2}$):** $\frac{0.3369}{0.9209} \approx \mathbf{0.3659}$ (36.6%)

---

#### Step 5: Compute LoRA Outputs & Combine Everything

Each LoRA adapter is a lightweight low-rank update $E_j^{\text{LoRA}}(x) = \frac{\alpha}{r} B_j A_j x$. Suppose for our token $x = [1.0, 2.0]$, the two chosen adapters produce:

* $E_1^{\text{LoRA}}(x) = \begin{bmatrix} 0.4 \\ 0.2 \end{bmatrix}$
* $E_2^{\text{LoRA}}(x) = \begin{bmatrix} -0.1 \\ 0.6 \end{bmatrix}$

We blend them using their renormalized weights:

$$\text{LoRA}(x) = 0.6341 \begin{bmatrix} 0.4 \\ 0.2 \end{bmatrix} + 0.3659 \begin{bmatrix} -0.1 \\ 0.6 \end{bmatrix} = \begin{bmatrix} 0.2536 \\ 0.1268 \end{bmatrix} + \begin{bmatrix} -0.0366 \\ 0.2195 \end{bmatrix} = \begin{bmatrix} \mathbf{0.2171} \\ \mathbf{0.3463} \end{bmatrix}$$

Finally, we add the frozen base MoE output, the new LoRA adaptation, and the original residual token $x$ together:

$$x_{\text{next}} = x + \text{MoE}(x) + \text{LoRA}(x)$$

$$x_{\text{next}} = \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix} + \begin{bmatrix} 2.1930 \\ 0.3450 \end{bmatrix} + \begin{bmatrix} 0.2171 \\ 0.3463 \end{bmatrix} = \begin{bmatrix} \mathbf{3.4101} \\ \mathbf{2.6913} \end{bmatrix}$$


In [ ]:
import numpy as np

x = np.array([1.0, 2.0])
z = np.array([3.0, 2.0, 0.4])
moe_out = np.array([2.19317574, 0.34470711])

W_h = np.array([[0.5, -0.2]])
W_l = np.array([
    [0.5, 0.1, -0.5, 1.0],
    [-0.2, 0.8, 0.0, 0.5],
    [0.1, -0.5, 1.0, -1.0]
])
K_L = 2

# Step 1. Compress the Token (h_c = W_h x)
h_c = W_h @ x
print("Step 1 - Compressed Token Signal (h_c):", np.round(h_c, 4))

# Step 2. Concatenate Both Channels ([z ; h_c])
rcp_input = np.concatenate([z, h_c])
print("Step 2 - Concatenated RCP Input [z ; h_c]:", np.round(rcp_input, 4))

# Step 3. Score the Global LoRA Pool (p_L = softmax(W_l * [z ; h_c]))
lora_logits = W_l @ rcp_input
exp_lora = np.exp(lora_logits)
p_L = exp_lora / np.sum(exp_lora)
print("Step 3 - Raw LoRA Logits:", np.round(lora_logits, 4))
print("Step 3 - Full LoRA Probabilities (p_L):", np.round(p_L, 4))

# Step 4. Top-K_L Selection & Renormalization (K_L = 2)
top_kl_indices = np.argsort(p_L)[::-1][:K_L]
top_kl_probs = p_L[top_kl_indices]
p_L_tilde = top_kl_probs / np.sum(top_kl_probs)
print("Step 4 - Selected LoRA Indices (0-based):", top_kl_indices)
print("Step 4 - Renormalized LoRA Weights (p_L_tilde):", np.round(p_L_tilde, 4))

# Step 5. Compute LoRA Outputs & Combine Everything
E1_lora_x = np.array([0.4, 0.2])
E2_lora_x = np.array([-0.1, 0.6])
lora_out = p_L_tilde[0] * E1_lora_x + p_L_tilde[1] * E2_lora_x
x_next_moe2 = x + moe_out + lora_out
print("Step 5 - Blended LoRA Output:", np.round(lora_out, 4))
print("Step 5 - Final Next Hidden State (MoE2-LoRA):", np.round(x_next_moe2, 4))


### Compact One-Line Tensor Chain: MoE²-LoRA Parallel Branch

#### 1. Parallel Routing-Conditioned Projection (RCP) & Global LoRA Branch
$$
\underbrace{x_{\text{flat}}}_{[T, D]}
\xrightarrow{W_h} \underbrace{h_c}_{[T, d_b]}
\xrightarrow{\text{Concat } [z ; h_c]} \underbrace{u_{\text{rcp}}}_{[T, N_E + d_b]}
\xrightarrow{W_l} \underbrace{z_L}_{[T, N_L]}
\xrightarrow{\text{Softmax}} \underbrace{p_L}_{[T, N_L]}
\xrightarrow{\text{Top-}K_L \text{ \& Renorm}} \underbrace{\tilde{p}_L}_{[T, K_L]}
\xrightarrow{\text{Dispatch \& } E_j^{\text{LoRA}}} \underbrace{E_j^{\text{LoRA}}(x_{\text{flat}})}_{[T_j, D]}
\xrightarrow{\sum \tilde{p}_{L,j} E_j^{\text{LoRA}}} \underbrace{\text{LoRA}(x)}_{[T, D]}
$$

#### 2. Final Combined Layer Output
$$
\underbrace{x}_{[B, S, D]} + \text{Reshape}\!\left(\underbrace{\text{MoE}(x)}_{[T, D]} + \underbrace{\text{LoRA}(x)}_{[T, D]}\right) \longrightarrow \underbrace{y}_{[B, S, D]}
$$

---

### Quick Dimension Key (MoE²-LoRA)
* **$[T, d_b]$**: Compressed auxiliary token representation ($d_b \ll D$).
* **$[T, N_E + d_b]$**: Concatenated RCP input combining frozen base router logits $z \in \mathbb{R}^{N_E}$ and compressed token signal $h_c \in \mathbb{R}^{d_b}$.
* **$[T, N_L]$**: Raw logits ($z_L$) and softmax probabilities ($p_L$) across all $N_L$ experts in the global shared LoRA pool.
* **$[T, K_L]$**: Renormalized gating weights ($\tilde{p}_L$) for the top $K_L$ selected global LoRA experts per token.
* **$[T_j, D]$**: Subset of $T_j$ tokens routed to global LoRA expert $j$.


### Walkthrough of the Compact One-Line Tensor Chain (MoE²-LoRA)

#### 1. Compressing the Token (Auxiliary Channel)
$$\underbrace{x_{\text{flat}}}_{[T, D]} \xrightarrow{W_h} \underbrace{h_c}_{[T, d_b]}$$
* **What happens:** The flattened input tokens are multiplied by a small down-projection matrix $W_h$, shrinking each token vector from the full hidden size $D$ down to a tiny bottleneck size $d_b$ (for example, $d_b = 16$ or $64$).
* **Why:** Projecting directly from the massive hidden dimension $D$ to hundreds of LoRA experts would waste too many parameters. This creates a lightweight, task-specific summary of each token.

#### 2. Fusing Base Priors & Task Signals (Concatenation)
$$\underbrace{h_c}_{[T, d_b]} \xrightarrow{\text{Concat } [z ; h_c]} \underbrace{u_{\text{rcp}}}_{[T, N_E + d_b]}$$
* **What happens:** The compressed token summary $h_c$ is glued side-by-side with $z$ ($[T, N_E]$), the raw routing scores already computed by the frozen base MoE router.
* **Why:** This is the core trick of **Routing-Conditioned Projection (RCP)**. It lets the LoRA router piggyback on the pretrained model's existing routing wisdom ($z$) while adjusting for the new fine-tuning task ($h_c$).

#### 3. Scoring the Global LoRA Library
$$\underbrace{u_{\text{rcp}}}_{[T, N_E + d_b]} \xrightarrow{W_l} \underbrace{z_L}_{[T, N_L]}$$
* **What happens:** This layer's trainable routing matrix $W_l$ multiplies the fused vector to produce raw preference scores ($z_L$) for all $N_L$ LoRA adapters in the model-wide shared pool.
* **Why:** Even though the $N_L$ LoRA adapters are shared across every floor (layer) of the model, each layer gets its own $W_l$ so it can choose whichever adapters fit its specific depth best.

#### 4. Converting Scores to Probabilities
$$\underbrace{z_L}_{[T, N_L]} \xrightarrow{\text{Softmax}} \underbrace{p_L}_{[T, N_L]}$$
* **What happens:** The raw scores across all $N_L$ adapters are passed through a softmax function.
* **Why:** This matches Equation (1) in the paper ($p_L = \text{softmax}(W_l \cdot [z ; h_c])$), turning raw scores into a clean probability distribution over the entire shared pool before pruning.

#### 5. Top-$K_L$ Selection & Renormalization
$$\underbrace{p_L}_{[T, N_L]} \xrightarrow{\text{Top-}K_L \text{ \& Renorm}} \underbrace{\tilde{p}_L}_{[T, K_L]}$$
* **What happens:** For each token, the model keeps only the $K_L$ highest probabilities and divides them by their sum so they add up to $1.0$ again ($\tilde{p}_L$).
* **Why:** Just like the base MoE layer, keeping only the top $K_L$ adapters ensures sparse, fast computation while preserving clean weighted averaging.

#### 6. Dispatching to Global LoRA Experts
$$\underbrace{\tilde{p}_L}_{[T, K_L]} \xrightarrow{\text{Dispatch \& } E_j^{\text{LoRA}}} \underbrace{E_j^{\text{LoRA}}(x_{\text{flat}})}_{[T_j, D]}$$
* **What happens:** Each token is routed to its $K_L$ chosen LoRA adapters from the global pool. Each active adapter $j$ runs its low-rank math $\frac{\alpha}{r} B_j A_j x$ on its assigned slice of $T_j$ tokens.
* **Why:** Low-rank bottleneck matrices ($A_j$ and $B_j$) compute lightweight task-specific adjustments in the original hidden dimension $D$ without touching the frozen base weights.

#### 7. Blending the LoRA Adjustments
$$\underbrace{E_j^{\text{LoRA}}(x_{\text{flat}})}_{[T_j, D]} \xrightarrow{\sum \tilde{p}_{L,j} E_j^{\text{LoRA}}} \underbrace{\text{LoRA}(x)}_{[T, D]}$$
* **What happens:** Each active LoRA adapter's output is multiplied by its renormalized weight $\tilde{p}_{L,j}$ from Step 5, and the results are summed for each token.
* **Why:** This merges the contributions of the selected LoRA adapters into a single adaptation vector $\text{LoRA}(x)$ for every token.

#### 8. Final Recombination & Residual Addition
$$\underbrace{x}_{[B, S, D]} + \text{Reshape}\!\left(\underbrace{\text{MoE}(x)}_{[T, D]} + \underbrace{\text{LoRA}(x)}_{[T, D]}\right) \longrightarrow \underbrace{y}_{[B, S, D]}$$
* **What happens:** The frozen base output $\text{MoE}(x)$ and the new adapter output $\text{LoRA}(x)$ are added together, reshaped back into 3D sequence format $[B, S, D]$, and added to the original input $x$.
* **Why:** This combines the pretrained model's general knowledge with the new task-specific specialization in a single seamless step.

---

### Quick Dimension Key (MoE²-LoRA)
* **$[T, d_b]$**: Compressed auxiliary token representation ($d_b \ll D$).
* **$[T, N_E + d_b]$**: Concatenated RCP input combining frozen base router logits $z \in \mathbb{R}^{N_E}$ and compressed token signal $h_c \in \mathbb{R}^{d_b}$.
* **$[T, N_L]$**: Raw logits ($z_L$) and softmax probabilities ($p_L$) across all $N_L$ experts in the global shared LoRA pool.
* **$[T, K_L]$**: Renormalized gating weights ($\tilde{p}_L$) for the top $K_L$ selected global LoRA experts per token.
* **$[T_j, D]$**: Subset of $T_j$ tokens routed to global LoRA expert $j$.


### Backpropagation Through MoE²-LoRA

Given the forward pass at layer $\ell$:
$$y^{(\ell)} = \text{MoE}^{(\ell)}(h^{(\ell)}) + \sum_{j \in S_h^{\text{LoRA}}} \tilde{p}_{L,j} \cdot E_j^{\text{LoRA}}(h^{(\ell)}), \quad E_j^{\text{LoRA}}(h) = \frac{\alpha}{r} B_j A_j h$$
and upstream gradient $\delta_y = \frac{\partial \mathcal{L}}{\partial y^{(\ell)}} \in \mathbb{R}^D$:

#### 1. Global Shared LoRA Expert Gradients ($A_j, B_j$)
For each active LoRA adapter $j \in S_h^{\text{LoRA}}$ at layer $\ell$:
$$\frac{\partial \mathcal{L}^{(\ell)}}{\partial B_j} = \tilde{p}_{L,j} \frac{\alpha}{r} \, \delta_y \big(A_j h^{(\ell)}\big)^\top, \qquad \frac{\partial \mathcal{L}^{(\ell)}}{\partial A_j} = \tilde{p}_{L,j} \frac{\alpha}{r} \big(B_j^\top \delta_y\big) \big(h^{(\ell)}\big)^\top$$

Accumulated across all $L$ layers sharing the global pool:
$$\frac{\partial \mathcal{L}_{\text{total}}}{\partial B_j} = \sum_{\ell=1}^{L} \mathbb{I}\!\left[j \in S_h^{\text{LoRA}, (\ell)}\right] \frac{\partial \mathcal{L}^{(\ell)}}{\partial B_j}, \qquad \frac{\partial \mathcal{L}_{\text{total}}}{\partial A_j} = \sum_{\ell=1}^{L} \mathbb{I}\!\left[j \in S_h^{\text{LoRA}, (\ell)}\right] \frac{\partial \mathcal{L}^{(\ell)}}{\partial A_j}$$

#### 2. Renormalization & Full Softmax Gradients ($p_L$ and $z_L$)
First compute the scalar alignment gradient for each active gate:
$$g_{\tilde{p}, j} = \delta_y^\top E_j^{\text{LoRA}}(h^{(\ell)}) \quad (j \in S_h^{\text{LoRA}})$$

Backpropagate through Top-$K_L$ renormalization ($Z_{\text{top}} = \sum_{k \in S_h^{\text{LoRA}}} p_{L,k}$):
$$\frac{\partial \mathcal{L}}{\partial p_{L,j}} = \begin{cases} \dfrac{1}{Z_{\text{top}}} \left( g_{\tilde{p}, j} - \sum_{k \in S_h^{\text{LoRA}}} \tilde{p}_{L,k} \, g_{\tilde{p}, k} \right) & \text{if } j \in S_h^{\text{LoRA}} \\ 0 & \text{if } j \notin S_h^{\text{LoRA}} \end{cases}$$

Backpropagate through the full softmax $p_L = \text{softmax}(z_L)$ across all $N_L$ experts:
$$\frac{\partial \mathcal{L}}{\partial z_{L,m}} = p_{L,m} \left( \frac{\partial \mathcal{L}}{\partial p_{L,m}} - \sum_{j \in S_h^{\text{LoRA}}} p_{L,j} \frac{\partial \mathcal{L}}{\partial p_{L,j}} \right)$$

#### 3. RCP Router ($W_l$) & Auxiliary Bottleneck ($W_h$) Gradients
With $u_{\text{rcp}} = [z ; h_c]$ and $\begin{bmatrix} \delta_z \\ \delta_{h_c} \end{bmatrix} = \big(W_l^{(\ell)}\big)^\top \frac{\partial \mathcal{L}}{\partial z_L}$:
$$\frac{\partial \mathcal{L}}{\partial W_l^{(\ell)}} = \left( \frac{\partial \mathcal{L}}{\partial z_L} \right) u_{\text{rcp}}^\top, \qquad \frac{\partial \mathcal{L}}{\partial W_h^{(\ell)}} = \delta_{h_c} \big(h^{(\ell)}\big)^\top$$

#### 4. Input Hidden State Gradient ($h^{(\ell)}$)
$$\frac{\partial \mathcal{L}}{\partial h^{(\ell)}} = \underbrace{\sum_{i \in S_h} g_i(h) \left( \frac{\partial E_i(h)}{\partial h} \right)^\top \delta_y + \left( \frac{\partial G(h)}{\partial h} \right)^\top \left( \frac{\partial \mathcal{L}_{\text{base}}}{\partial z} + \delta_z \right)}_{\text{Frozen Base Paths}} + \underbrace{\sum_{j \in S_h^{\text{LoRA}}} \tilde{p}_{L,j} \frac{\alpha}{r} A_j^\top B_j^\top \delta_y + \big(W_h^{(\ell)}\big)^\top \delta_{h_c}}_{\text{Trainable MoE}^2\text{-LoRA Paths}}$$


### Pre-training vs. MoE²-LoRA Fine-Tuning

**Basic pre-training is done first** on the base MoE model, and then **MoE²-LoRA fine-tunes only the newly added adapter and routing parameters** while keeping the entire pretrained model frozen.

#### 1. Stage 1: Standard Base Pre-training (Done Beforehand)
You start with an already pretrained Mixture-of-Experts backbone—such as OLMoE-1B-7B, DeepSeek-V2-Lite, or Qwen3-30B-A3B. During this stage, the base model's router $G(\cdot)$ and experts $\{E_i\}$ have already learned general language patterns and expert specializations.

#### 2. Stage 2: MoE²-LoRA Fine-Tuning (Adapter-Only Training)
When adapting the model to a downstream task (like math, coding, or medical VQA), you freeze 100% of the pretrained backbone—including the base MoE experts, the base MoE router, and all attention/embedding layers.

#### 3. Exactly Which Parameters Are Trained?
During MoE²-LoRA fine-tuning, gradient updates apply exclusively to the lightweight components added to the MoE blocks:
* **The Global Shared LoRA Expert Pool ($\{A_j, B_j\}_{j=1}^{N_L}$):** The low-rank adapter matrices shared across all MoE layers.
* **The Per-Layer RCP Routing Matrices ($W_l$):** The learnable projections that map the combined routing signals into scores for the LoRA pool.
* **The Per-Layer Auxiliary Bottleneck Projections ($W_h$):** The small matrices that compress the hidden state $h$ into the auxiliary channel $h_c$.

#### 4. Why This Order Matters
MoE²-LoRA relies directly on the **pretrained base router's logits ($z = G(h)$)** as its primary routing signal. Because the base model is already pretrained and frozen, those router logits provide a stable, built-in map of token types from the very first step of fine-tuning rather than forcing the adapter router to learn from scratch.


### Explicit Low-Rank Matrix Calculation ($E_j^{\text{LoRA}}(x) = \frac{\alpha}{r} B_j A_j x$)

To calculate the adapter outputs explicitly rather than treating them as black-box vectors, we define concrete dimensions and weights for the low-rank matrices $A_j$ and $B_j$.

#### 1. Setting Dimensions and Scaling Factor
For our toy token $x = \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix}$ with hidden dimension $D = 2$:
* **Rank ($r$):** $1$ (since $r \ll D$).
* **Alpha ($\alpha$):** $1.0$ (giving a scaling ratio $\frac{\alpha}{r} = \frac{1.0}{1} = 1.0$).
* **Down-projection matrix ($A_j$):** Shape $[r, D] = [1, 2]$.
* **Up-projection matrix ($B_j$):** Shape $[D, r] = [2, 1]$.

---

#### 2. Concrete Matrices for Adapter 1 ($E_1^{\text{LoRA}}$)
Suppose Adapter 1 has the following learned weights:
$$A_1 = \begin{bmatrix} 0.2 & 0.1 \end{bmatrix}, \qquad B_1 = \begin{bmatrix} 1.0 \\ 0.5 \end{bmatrix}$$

Multiplying step by step with $x = \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix}$:
1. **Down-project ($A_1 x$):**
   $$A_1 x = \begin{bmatrix} 0.2 & 0.1 \end{bmatrix} \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix} = (0.2 \times 1.0) + (0.1 \times 2.0) = \mathbf{0.4}$$
2. **Up-project ($B_1(A_1 x)$):**
   $$B_1 (0.4) = \begin{bmatrix} 1.0 \\ 0.5 \end{bmatrix} (0.4) = \begin{bmatrix} 0.4 \\ 0.2 \end{bmatrix}$$
3. **Apply Scaling ($\frac{\alpha}{r} = 1.0$):**
   $$E_1^{\text{LoRA}}(x) = 1.0 \cdot \begin{bmatrix} 0.4 \\ 0.2 \end{bmatrix} = \begin{bmatrix} \mathbf{0.4} \\ \mathbf{0.2} \end{bmatrix}$$

---

#### 3. Concrete Matrices for Adapter 2 ($E_2^{\text{LoRA}}$)
Suppose Adapter 2 has the following learned weights:
$$A_2 = \begin{bmatrix} 0.5 & -0.2 \end{bmatrix}, \qquad B_2 = \begin{bmatrix} -1.0 \\ 6.0 \end{bmatrix}$$

Multiplying step by step with $x = \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix}$:
1. **Down-project ($A_2 x$):**
   $$A_2 x = \begin{bmatrix} 0.5 & -0.2 \end{bmatrix} \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix} = (0.5 \times 1.0) + (-0.2 \times 2.0) = \mathbf{0.1}$$
2. **Up-project ($B_2(A_2 x)$):**
   $$B_2 (0.1) = \begin{bmatrix} -1.0 \\ 6.0 \end{bmatrix} (0.1) = \begin{bmatrix} -0.1 \\ 0.6 \end{bmatrix}$$
3. **Apply Scaling ($\frac{\alpha}{r} = 1.0$):**
   $$E_2^{\text{LoRA}}(x) = 1.0 \cdot \begin{bmatrix} -0.1 \\ 0.6 \end{bmatrix} = \begin{bmatrix} \mathbf{-0.1} \\ \mathbf{0.6} \end{bmatrix}$$

---

#### Key Takeaway on Rank-1 Constraint
Because $r = 1$ in this toy example, $A_j x$ collapses the token into a single scalar value (`0.4` for Adapter 1 and `0.1` for Adapter 2), making every output vector a scaled multiple of $B_j$. In the actual paper, ranks like $r = 16$ or $r = 32$ are used so $A_j x$ produces a 16- or 32-element feature vector before $B_j$ projects it back up.


### Complete Left-to-Right MoE Transformer Block Pipeline ($X_{\text{in}} \to \dots \to Y$)

#### Stage A: Multi-Head Self-Attention Sub-Layer (Sequence Mixing)
$$
\underbrace{X_{\text{in}}}_{[B, S, D]}
\xrightarrow{\operatorname{Norm}_1} \underbrace{\tilde{X}}_{[B, S, D]}
\xrightarrow{\begin{subarray}{c} Q = \tilde{X} W_Q^\top \\ K = \tilde{X} W_K^\top \\ V = \tilde{X} W_V^\top \end{subarray}} \underbrace{(Q, K, V)}_{[B, H, S, d_h]}
\xrightarrow{\operatorname{Softmax}\!\left(\frac{Q K^\top}{\sqrt{d_h}} + M\right) V} \underbrace{A}_{[B, H, S, d_h]}
\xrightarrow{\operatorname{Concat}_H(\cdot) W_O^\top} \underbrace{\operatorname{Attn}(\tilde{X})}_{[B, S, D]}
\xrightarrow{+ \, X_{\text{in}}} \underbrace{X}_{[B, S, D]}
$$

#### Stage B: Sparse Top-$K$ MoE Sub-Layer (Token-Wise Transformation)
$$
\underbrace{X}_{[B, S, D]}
\xrightarrow{\operatorname{Norm}_2 \to \operatorname{Reshape}_{T \times D}} \underbrace{X_{\text{flat}}}_{[T, D]}
\xrightarrow{\begin{subarray}{c} G = \operatorname{Softmax}_{\text{row}}(\operatorname{TopK}_{\text{row}}(X_{\text{flat}} W_g^\top, K_{\text{exp}})) \\ E_i = \operatorname{MLP}_i(X_{\text{flat}}) \end{subarray}} \underbrace{\sum_{i=1}^{N} \operatorname{Diag}(G_{:, i}) E_i}_{[T, D]}
\xrightarrow{\operatorname{Reshape}_{B \times S \times D}} \underbrace{\operatorname{MoE}(\hat{X})}_{[B, S, D]}
\xrightarrow{+ \, X} \underbrace{Y}_{[B, S, D]}
$$

### Tensor Dimension & Matrix Key

* **$B, S, D$**: Batch size, Sequence length, and Model hidden dimension.
* **$H, d_h$**: Number of attention heads and per-head dimension ($D = H \cdot d_h$).
* **$W_Q, W_K, W_V, W_O \in \mathbb{R}^{D \times D}$**: Query, Key, Value, and Output projection matrices (split across $H$ heads).
* **$M \in \mathbb{R}^{S \times S}$**: Causal attention mask preventing tokens from attending to future positions.
* **$T = B \cdot S$**: Flattened token count entering the token-wise MoE router.
* **$W_g \in \mathbb{R}^{N \times D}$**: MoE router matrix scoring $N$ experts per token, picking top $K_{\text{exp}}$.
* **$\operatorname{Diag}(G_{:, i}) \in \mathbb{R}^{T \times T}$**: Diagonal gating matrix weighting expert $i$'s output $E_i \in \mathbb{R}^{T \times D}$.


### Left-to-Right Stacked Per-Layer MoE Structural Tensor Pipeline ($X \to \dots \to Y$)

#### 1. Full Model Depth Flow (Across All $L$ Layers)
$$
\underbrace{X}_{[B, S]}
\xrightarrow{\text{Embed}} \underbrace{H^{(0)}}_{[B, S, D]}
\xrightarrow{\text{Layer } 1} \underbrace{H^{(1)}}_{[B, S, D]}
\xrightarrow{\text{Layer } 2} \underbrace{H^{(2)}}_{[B, S, D]}
\xrightarrow{\cdots} \underbrace{H^{(L-1)}}_{[B, S, D]}
\xrightarrow{\text{Layer } L} \underbrace{H^{(L)}}_{[B, S, D]}
\xrightarrow{\text{Norm} \to W_{\text{head}}^\top} \underbrace{Y}_{[B, S, |V|]}
$$

#### 2. Inside Any Single Layer $\ell \in \{1, \dots, L\}$ ($H^{(\ell-1)} \to H^{(\ell)}$)
$$
\underbrace{H^{(\ell-1)}}_{[B, S, D]}
\xrightarrow{\operatorname{Attn}^{(\ell)} + H^{(\ell-1)}} \underbrace{\tilde{H}^{(\ell)}}_{[B, S, D]}
\xrightarrow{\operatorname{Reshape}_{T \times D}} \underbrace{\tilde{H}_{\text{flat}}^{(\ell)}}_{[T, D]}
\xrightarrow{\begin{subarray}{c} G^{(\ell)} = \operatorname{Softmax}_{\text{row}}(\operatorname{TopK}_{\text{row}}(\tilde{H}_{\text{flat}}^{(\ell)} (W_g^{(\ell)})^\top, K)) \\ E_i^{(\ell)} = \operatorname{MLP}_i^{(\ell)}(\tilde{H}_{\text{flat}}^{(\ell)}) \end{subarray}} \underbrace{\sum_{i=1}^{N_E} \operatorname{Diag}(G_{:, i}^{(\ell)}) E_i^{(\ell)}}_{[T, D]}
\xrightarrow{\operatorname{Reshape}_{B \times S \times D} + \tilde{H}^{(\ell)}} \underbrace{H^{(\ell)}}_{[B, S, D]}
$$

---

### Key Takeaways on Layer Indexing ($\ell$)
* **Token Evolution ($\tilde{H}^{(\ell)}$):** Self-Attention $\operatorname{Attn}^{(\ell)}$ updates every token with new sequence context at each depth before routing happens.
* **Independent Routers ($W_g^{(\ell)}$):** Each layer $\ell$ has its own router weight matrix to make fresh Top-$K$ routing decisions on $\tilde{H}_{\text{flat}}^{(\ell)}$.
* **Isolated Expert Pools ($E_i^{(\ell)}$):** In a standard base MoE, each layer $\ell$ has its own separate pool of $N_E$ expert MLPs (whereas MoE²-LoRA replaces per-layer adapter pools with one shared global pool across all $\ell$).


### 2-Layer Stacked MoE Numerical Walkthrough ($L = 2$)

This walkthrough traces how a single token vector $x = \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix}$ ($D = 2$) travels sequentially through **Layer 1** and **Layer 2**. Because each layer has its own attention block, its own router matrix $W_g^{(\ell)}$, and its own isolated pool of expert MLPs $E_i^{(\ell)}$, the token evolves after Layer 1 and triggers a completely different routing choice in Layer 2.

---

#### Setup: Initial Token State
Our token enters **Layer 1** with hidden dimension $D = 2$:
$$h^{(0)} = \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix}$$

---

#### Layer 1: Shallow Processing ($\ell = 1$)

##### 1A. Layer 1 Self-Attention + Residual
Suppose Layer 1's self-attention block gathers context from neighboring words and adds $\text{Attn}^{(1)}(h^{(0)}) = \begin{bmatrix} 0.0 \\ 0.0 \end{bmatrix}$:
$$\tilde{h}^{(1)} = h^{(0)} + \text{Attn}^{(1)}(h^{(0)}) = \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix}$$

##### 1B. Layer 1 Router ($W_g^{(1)}$) Picks Experts
Layer 1 uses **its own router matrix** $W_g^{(1)}$ to score Layer 1's 3 experts:
$$W_g^{(1)} = \begin{bmatrix} 1.0 & 1.0 \\ -1.0 & 1.5 \\ 0.2 & 0.1 \end{bmatrix} \implies z^{(1)} = W_g^{(1)} \tilde{h}^{(1)} = \begin{bmatrix} \mathbf{3.0} \\ \mathbf{2.0} \\ 0.4 \end{bmatrix}$$
* **Top-2 Selection:** Picks **Expert 1** ($3.0$) and **Expert 2** ($2.0$).
* **Softmax Gating Weights:** $g_1^{(1)} \approx \mathbf{0.731}$, $g_2^{(1)} \approx \mathbf{0.269}$.

##### 1C. Run Layer 1's Experts & Residual Add
Layer 1 runs **its own isolated experts** $E_1^{(1)}$ and $E_2^{(1)}$, producing $E_1^{(1)}(\tilde{h}^{(1)}) = \begin{bmatrix} 3.0 \\ -1.0 \end{bmatrix}$ and $E_2^{(1)}(\tilde{h}^{(1)}) = \begin{bmatrix} 0.0 \\ 4.0 \end{bmatrix}$:
$$\text{MoE}^{(1)}(\tilde{h}^{(1)}) = 0.731 \begin{bmatrix} 3.0 \\ -1.0 \end{bmatrix} + 0.269 \begin{bmatrix} 0.0 \\ 4.0 \end{bmatrix} = \begin{bmatrix} 2.193 \\ 0.345 \end{bmatrix}$$

Adding the residual stream gives the updated token leaving Layer 1:
$$h^{(1)} = \tilde{h}^{(1)} + \text{MoE}^{(1)}(\tilde{h}^{(1)}) = \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix} + \begin{bmatrix} 2.193 \\ 0.345 \end{bmatrix} = \begin{bmatrix} \mathbf{3.193} \\ \mathbf{2.345} \end{bmatrix}$$

---

#### Layer 2: Deeper Processing ($\ell = 2$)

Now the evolved token $h^{(1)} = \begin{bmatrix} 3.193 \\ 2.345 \end{bmatrix}$ enters **Layer 2**.

##### 2A. Layer 2 Self-Attention + Residual
Layer 2's attention block refines the token further, adding $\text{Attn}^{(2)}(h^{(1)}) = \begin{bmatrix} -0.193 \\ 0.655 \end{bmatrix}$:
$$\tilde{h}^{(2)} = h^{(1)} + \text{Attn}^{(2)}(h^{(1)}) = \begin{bmatrix} 3.193 \\ 2.345 \end{bmatrix} + \begin{bmatrix} -0.193 \\ 0.655 \end{bmatrix} = \begin{bmatrix} \mathbf{3.000} \\ \mathbf{3.000} \end{bmatrix}$$

##### 2B. Layer 2 Router ($W_g^{(2)}$) Makes a Brand-New Decision
Layer 2 has **its own distinct router matrix** $W_g^{(2)}$ trained for deeper features:
$$W_g^{(2)} = \begin{bmatrix} -0.5 & 0.2 \\ 0.4 & 0.6 \\ 0.8 & 0.5 \end{bmatrix}$$

Multiplying $W_g^{(2)} \tilde{h}^{(2)}$ with $\tilde{h}^{(2)} = \begin{bmatrix} 3.0 \\ 3.0 \end{bmatrix}$:
* **Layer 2 Expert 1:** $(-0.5 \times 3.0) + (0.2 \times 3.0) = -0.9$
* **Layer 2 Expert 2:** $(0.4 \times 3.0) + (0.6 \times 3.0) = \mathbf{3.0}$
* **Layer 2 Expert 3:** $(0.8 \times 3.0) + (0.5 \times 3.0) = \mathbf{3.9}$

* **Top-2 Selection:** Because both the token vector and the router changed, Layer 2 drops Expert 1 and instead picks **Expert 3** ($3.9$) and **Expert 2** ($3.0$).
* **Softmax Gating Weights:**
  $$g_3^{(2)} = \frac{e^{3.9}}{e^{3.9} + e^{3.0}} \approx \mathbf{0.711}, \qquad g_2^{(2)} = \frac{e^{3.0}}{e^{3.9} + e^{3.0}} \approx \mathbf{0.289}$$

##### 2C. Run Layer 2's Separate Experts & Final Residual Add
Layer 2 runs **its own isolated experts** $E_3^{(2)}$ and $E_2^{(2)}$ (which hold completely different weights from Layer 1's experts). Suppose $E_3^{(2)}(\tilde{h}^{(2)}) = \begin{bmatrix} 1.0 \\ -2.0 \end{bmatrix}$ and $E_2^{(2)}(\tilde{h}^{(2)}) = \begin{bmatrix} -1.0 \\ 1.0 \end{bmatrix}$:
$$\text{MoE}^{(2)}(\tilde{h}^{(2)}) = 0.711 \begin{bmatrix} 1.0 \\ -2.0 \end{bmatrix} + 0.289 \begin{bmatrix} -1.0 \\ 1.0 \end{bmatrix} = \begin{bmatrix} 0.422 \\ -1.133 \end{bmatrix}$$

Adding the residual stream produces the output token leaving Layer 2:
$$h^{(2)} = \tilde{h}^{(2)} + \text{MoE}^{(2)}(\tilde{h}^{(2)}) = \begin{bmatrix} 3.000 \\ 3.000 \end{bmatrix} + \begin{bmatrix} 0.422 \\ -1.133 \end{bmatrix} = \begin{bmatrix} \mathbf{3.422} \\ \mathbf{1.867} \end{bmatrix}$$

---

#### Summary of What Changed Across Layers

| Stage | Token Entering MoE ($\tilde{h}^{(\ell)}$) | Active Experts Chosen | Token Leaving Layer ($h^{(\ell)}$) |
| :--- | :--- | :--- | :--- |
| **Layer 1** | $[1.000, \, 2.000]$ | **Expert 1** (73.1%) + **Expert 2** (26.9%) | $[3.193, \, 2.345]$ |
| **Layer 2** | $[3.000, \, 3.000]$ | **Expert 3** (71.1%) + **Expert 2** (28.9%) | $[3.422, \, 1.867]$ |

*(In **MoE²-LoRA**, Layer 1 and Layer 2 would each keep their own separate RCP routers $W_l^{(1)}$ and $W_l^{(2)}$, allowing both layers to query the exact same **Global Shared LoRA Pool** while still picking different adapters at each depth.)*


### Left-to-Right MoE²-LoRA Structural Tensor Pipeline ($X \to \dots \to Y$)

#### 1. Full Model Depth Flow (Frozen Backbone + Shared Global Pool)
$$
\underbrace{X}_{[B, S]}
\xrightarrow{\text{Embed}} \underbrace{H^{(0)}}_{[B, S, D]}
\xrightarrow[\text{Pool } \{A_j, B_j\}_{j=1}^{N_L}]{\text{MoE}^2\text{-Layer } 1} \underbrace{H^{(1)}}_{[B, S, D]}
\xrightarrow[\text{Pool } \{A_j, B_j\}_{j=1}^{N_L}]{\text{MoE}^2\text{-Layer } 2} \underbrace{H^{(2)}}_{[B, S, D]}
\xrightarrow{\cdots} \underbrace{H^{(L-1)}}_{[B, S, D]}
\xrightarrow[\text{Pool } \{A_j, B_j\}_{j=1}^{N_L}]{\text{MoE}^2\text{-Layer } L} \underbrace{H^{(L)}}_{[B, S, D]}
\xrightarrow{\text{Norm} \to W_{\text{head}}^\top} \underbrace{Y}_{[B, S, |V|]}
$$

#### 2. Inside Any Single Layer $\ell \in \{1, \dots, L\}$ ($H^{(\ell-1)} \to H^{(\ell)}$)

**Step A: Frozen Self-Attention & Flatten**
$$
\underbrace{H^{(\ell-1)}}_{[B, S, D]}
\xrightarrow{\operatorname{Attn}^{(\ell)}_{\text{frozen}} + H^{(\ell-1)}} \underbrace{\tilde{H}^{(\ell)}}_{[B, S, D]}
\xrightarrow{\operatorname{Reshape}_{T \times D}} \underbrace{\tilde{H}_{\text{flat}}^{(\ell)}}_{[T, D]}
$$

**Step B1: Frozen Base MoE Branch (Base Output + Reusable Logits $Z^{(\ell)}$)**
$$
\underbrace{\tilde{H}_{\text{flat}}^{(\ell)}}_{[T, D]}
\xrightarrow{Z^{(\ell)} = \tilde{H}_{\text{flat}}^{(\ell)} (W_g^{(\ell)})^\top} \underbrace{Z^{(\ell)}}_{[T, N_E]}
\xrightarrow{\begin{subarray}{c} G^{(\ell)} = \operatorname{Softmax}_{\text{row}}(\operatorname{TopK}_{\text{row}}(Z^{(\ell)}, K)) \\ E_i^{(\ell)} = \operatorname{MLP}_i^{(\ell)}(\tilde{H}_{\text{flat}}^{(\ell)}) \end{subarray}} \underbrace{\sum_{i=1}^{N_E} \operatorname{Diag}(G_{:, i}^{(\ell)}) E_i^{(\ell)}}_{[T, D]} = \operatorname{MoE}^{(\ell)}(\tilde{H}_{\text{flat}}^{(\ell)})
$$

**Step B2: Parallel Trainable MoE²-LoRA Branch (Reuses $Z^{(\ell)}$ & Queries Global Pool)**
$$
\underbrace{\tilde{H}_{\text{flat}}^{(\ell)}}_{[T, D]}
\xrightarrow{H_c^{(\ell)} = \tilde{H}_{\text{flat}}^{(\ell)} (W_h^{(\ell)})^\top} \underbrace{H_c^{(\ell)}}_{[T, d_b]}
\xrightarrow{\operatorname{Concat}[Z^{(\ell)} ; H_c^{(\ell)}]} \underbrace{U_{\text{rcp}}^{(\ell)}}_{[T, N_E + d_b]}
\xrightarrow{P_L^{(\ell)} = \operatorname{Softmax}_{\text{row}}(U_{\text{rcp}}^{(\ell)} (W_l^{(\ell)})^\top)} \underbrace{P_L^{(\ell)}}_{[T, N_L]}
\xrightarrow{\operatorname{Renorm}(\operatorname{TopK}_{\text{row}}(P_L^{(\ell)}, K_L))} \underbrace{\tilde{P}_L^{(\ell)}}_{[T, K_L]}
$$

$$
\underbrace{\tilde{P}_L^{(\ell)}}_{[T, K_L]}
\xrightarrow[\text{Global Pool } (A_j, B_j)]{E_j^{\text{LoRA}} = \frac{\alpha}{r} \tilde{H}_{\text{flat}}^{(\ell)} A_j^\top B_j^\top} \underbrace{\sum_{j=1}^{N_L} \operatorname{Diag}(\tilde{P}_{L, :, j}^{(\ell)}) E_j^{\text{LoRA}}}_{[T, D]} = \operatorname{LoRA}^{(\ell)}(\tilde{H}_{\text{flat}}^{(\ell)})
$$

**Step C: Parallel Recombination & Residual Addition**
$$
\underbrace{\operatorname{MoE}^{(\ell)}(\tilde{H}_{\text{flat}}^{(\ell)})}_{[T, D]} + \underbrace{\operatorname{LoRA}^{(\ell)}(\tilde{H}_{\text{flat}}^{(\ell)})}_{[T, D]}
\xrightarrow{\operatorname{Reshape}_{B \times S \times D} + \tilde{H}^{(\ell)}} \underbrace{H^{(\ell)}}_{[B, S, D]}
$$

### What Changed From the Base MoE Pipeline (Superscript $\ell$ vs. Global)

* **Layer-Specific Trainable Routers ($W_h^{(\ell)}, W_l^{(\ell)}$):** Keep the $(\ell)$ superscript because each layer learns its own bottleneck projection $W_h^{(\ell)} \in \mathbb{R}^{d_b \times D}$ and RCP router $W_l^{(\ell)} \in \mathbb{R}^{N_L \times (N_E + d_b)}$.
* **Global Shared LoRA Experts ($A_j, B_j$):** Drop the $(\ell)$ superscript because all $N_L$ low-rank adapter pairs ($A_j \in \mathbb{R}^{r \times D}, B_j \in \mathbb{R}^{D \times r}$) live in **one global pool** reused across every layer $\ell \in \{1, \dots, L\}$.
* **Cross-Branch Coupling ($Z^{(\ell)}$):** The frozen base router logits $Z^{(\ell)} \in \mathbb{R}^{T \times N_E}$ computed in Step B1 are fed directly into Step B2's concatenation $\operatorname{Concat}[Z^{(\ell)} ; H_c^{(\ell)}]$.


### 2-Layer MoE²-LoRA Numerical Walkthrough ($L = 2$)

This walkthrough traces how a single token vector $x = \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix}$ ($D = 2$) travels through a **2-layer MoE²-LoRA model** ($L = 2$). Unlike a standard base MoE where every layer has its own isolated expert pool, **Layer 1** and **Layer 2** share **one Global LoRA Pool** of $N_L = 3$ rank-1 adapters ($r=1, \alpha=1.0$), while using their own separate **Routing-Conditioned Projection (RCP)** routers ($W_l^{(1)}, W_h^{(1)}$ and $W_l^{(2)}, W_h^{(2)}$) to pick top-$K_L = 2$ adapters at each depth.

---

#### Setup: The Global Shared LoRA Pool (Shared by Both Layers)
Suppose our single central library holds $N_L = 3$ rank-1 LoRA adapters ($E_j^{\text{LoRA}}(h) = \frac{\alpha}{r} B_j A_j h$, with $\frac{\alpha}{r} = 1.0$):
* **Adapter 1:** $A_1 = \begin{bmatrix} 0.2 & 0.1 \end{bmatrix}, \quad B_1 = \begin{bmatrix} 1.0 \\ 0.5 \end{bmatrix}$
* **Adapter 2:** $A_2 = \begin{bmatrix} 0.5 & -0.2 \end{bmatrix}, \quad B_2 = \begin{bmatrix} -1.0 \\ 6.0 \end{bmatrix}$
* **Adapter 3:** $A_3 = \begin{bmatrix} 0.1 & 0.3 \end{bmatrix}, \quad B_3 = \begin{bmatrix} 0.5 \\ -0.5 \end{bmatrix}$

---

#### Layer 1: Shallow Processing ($\ell = 1$)

Our token enters Layer 1's MoE²-LoRA block as $\tilde{h}^{(1)} = \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix}$.

##### 1A. Frozen Base MoE Branch
* **Base Router Logits ($z^{(1)}$):** $\begin{bmatrix} 3.0, & 2.0, & 0.4 \end{bmatrix}$ (selecting Base Experts 1 & 2).
* **Frozen Base MoE Output:** $\text{MoE}^{(1)}(\tilde{h}^{(1)}) = \begin{bmatrix} 2.1930 \\ 0.3450 \end{bmatrix}$.

##### 1B. Layer 1 RCP Router Queries the Global Pool
1. **Auxiliary Bottleneck ($h_c^{(1)} = W_h^{(1)} \tilde{h}^{(1)}$):**
   $$h_c^{(1)} = \begin{bmatrix} 0.5 & -0.2 \end{bmatrix} \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix} = \mathbf{0.1}$$
2. **Concatenate Channels ($[z^{(1)} ; h_c^{(1)}]$):**
   $$u_{\text{rcp}}^{(1)} = \begin{bmatrix} 3.0, & 2.0, & 0.4, & 0.1 \end{bmatrix}^T$$
3. **Score Global Pool ($p_L^{(1)} = \text{softmax}(W_l^{(1)} u_{\text{rcp}}^{(1)})$):**
   $$W_l^{(1)} = \begin{bmatrix} 0.5 & 0.1 & -0.5 & 1.0 \\ -0.2 & 0.8 & 0.0 & 0.5 \\ 0.1 & -0.5 & 1.0 & -1.0 \end{bmatrix} \implies W_l^{(1)} u_{\text{rcp}}^{(1)} = \begin{bmatrix} 1.60 \\ 1.05 \\ -0.40 \end{bmatrix} \xrightarrow{\text{softmax}} p_L^{(1)} = \begin{bmatrix} \mathbf{0.5840}, & \mathbf{0.3369}, & 0.0790 \end{bmatrix}$$
4. **Top-2 Selection & Renormalization:** Layer 1 selects **Global Adapter 1** and **Global Adapter 2**:
   $$\tilde{p}_{L,1}^{(1)} = \frac{0.5840}{0.5840 + 0.3369} = \mathbf{0.6341}, \qquad \tilde{p}_{L,2}^{(1)} = \frac{0.3369}{0.5840 + 0.3369} = \mathbf{0.3659}$$

##### 1C. Compute Global Adapters & Combine Layer 1 Output
Evaluating **Global Adapter 1** and **Global Adapter 2** on $\tilde{h}^{(1)} = \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix}$:
* $E_1^{\text{LoRA}}(\tilde{h}^{(1)}) = \begin{bmatrix} 1.0 \\ 0.5 \end{bmatrix} \left( \begin{bmatrix} 0.2 & 0.1 \end{bmatrix} \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix} \right) = \begin{bmatrix} 1.0 \\ 0.5 \end{bmatrix} (0.4) = \begin{bmatrix} 0.4 \\ 0.2 \end{bmatrix}$
* $E_2^{\text{LoRA}}(\tilde{h}^{(1)}) = \begin{bmatrix} -1.0 \\ 6.0 \end{bmatrix} \left( \begin{bmatrix} 0.5 & -0.2 \end{bmatrix} \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix} \right) = \begin{bmatrix} -1.0 \\ 6.0 \end{bmatrix} (0.1) = \begin{bmatrix} -0.1 \\ 0.6 \end{bmatrix}$

Blending them and adding to the frozen base output and residual stream $\tilde{h}^{(1)}$:
$$\text{LoRA}^{(1)}(\tilde{h}^{(1)}) = 0.6341 \begin{bmatrix} 0.4 \\ 0.2 \end{bmatrix} + 0.3659 \begin{bmatrix} -0.1 \\ 0.6 \end{bmatrix} = \begin{bmatrix} \mathbf{0.2171} \\ \mathbf{0.3463} \end{bmatrix}$$

$$h^{(1)} = \tilde{h}^{(1)} + \text{MoE}^{(1)}(\tilde{h}^{(1)}) + \text{LoRA}^{(1)}(\tilde{h}^{(1)}) = \begin{bmatrix} 1.0 \\ 2.0 \end{bmatrix} + \begin{bmatrix} 2.1930 \\ 0.3450 \end{bmatrix} + \begin{bmatrix} 0.2171 \\ 0.3463 \end{bmatrix} = \begin{bmatrix} \mathbf{3.4101} \\ \mathbf{2.6913} \end{bmatrix}$$

---

#### Layer 2: Deeper Processing ($\ell = 2$)

Suppose after Layer 2's frozen self-attention block refines $h^{(1)}$, the token enters Layer 2's MoE²-LoRA block as $\tilde{h}^{(2)} = \begin{bmatrix} 3.0 \\ 3.0 \end{bmatrix}$.

##### 2A. Frozen Base MoE Branch
* **Base Router Logits ($z^{(2)}$):** $\begin{bmatrix} -0.9, & 3.0, & 3.9 \end{bmatrix}$ (selecting Base Experts 3 & 2).
* **Frozen Base MoE Output:** $\text{MoE}^{(2)}(\tilde{h}^{(2)}) = \begin{bmatrix} 0.4220 \\ -1.1330 \end{bmatrix}$.

##### 2B. Layer 2's Separate RCP Router Queries the *Same* Global Pool
1. **Auxiliary Bottleneck ($h_c^{(2)} = W_h^{(2)} \tilde{h}^{(2)}$):**
   $$h_c^{(2)} = \begin{bmatrix} -0.1 & 0.4 \end{bmatrix} \begin{bmatrix} 3.0 \\ 3.0 \end{bmatrix} = \mathbf{0.9}$$
2. **Concatenate Channels ($[z^{(2)} ; h_c^{(2)}]$):**
   $$u_{\text{rcp}}^{(2)} = \begin{bmatrix} -0.9, & 3.0, & 3.9, & 0.9 \end{bmatrix}^T$$
3. **Score Global Pool ($p_L^{(2)} = \text{softmax}(W_l^{(2)} u_{\text{rcp}}^{(2)})$):**
   Layer 2 uses its own router matrix $W_l^{(2)}$:
   $$W_l^{(2)} = \begin{bmatrix} 0.1 & -0.2 & 0.0 & 0.2 \\ 0.0 & 0.4 & 0.1 & 0.5 \\ -0.3 & 0.2 & 0.5 & 0.1 \end{bmatrix} \implies W_l^{(2)} u_{\text{rcp}}^{(2)} = \begin{bmatrix} -0.51 \\ 2.04 \\ 2.91 \end{bmatrix} \xrightarrow{\text{softmax}} p_L^{(2)} = \begin{bmatrix} 0.0225, & \mathbf{0.2885}, & \mathbf{0.6890} \end{bmatrix}$$
4. **Top-2 Selection & Renormalization:** Layer 2 picks **Global Adapter 3** and **Global Adapter 2** (reusing Global Adapter 2 from Layer 1 while swapping out Adapter 1 for Adapter 3):
   $$\tilde{p}_{L,3}^{(2)} = \frac{0.6890}{0.6890 + 0.2885} = \mathbf{0.7049}, \qquad \tilde{p}_{L,2}^{(2)} = \frac{0.2885}{0.6890 + 0.2885} = \mathbf{0.2951}$$

##### 2C. Compute Reused Global Adapters & Combine Layer 2 Output
Evaluating the shared **Global Adapter 3** and **Global Adapter 2** on $\tilde{h}^{(2)} = \begin{bmatrix} 3.0 \\ 3.0 \end{bmatrix}$:
* $E_3^{\text{LoRA}}(\tilde{h}^{(2)}) = \begin{bmatrix} 0.5 \\ -0.5 \end{bmatrix} \left( \begin{bmatrix} 0.1 & 0.3 \end{bmatrix} \begin{bmatrix} 3.0 \\ 3.0 \end{bmatrix} \right) = \begin{bmatrix} 0.5 \\ -0.5 \end{bmatrix} (1.2) = \begin{bmatrix} 0.60 \\ -0.60 \end{bmatrix}$
* $E_2^{\text{LoRA}}(\tilde{h}^{(2)}) = \begin{bmatrix} -1.0 \\ 6.0 \end{bmatrix} \left( \begin{bmatrix} 0.5 & -0.2 \end{bmatrix} \begin{bmatrix} 3.0 \\ 3.0 \end{bmatrix} \right) = \begin{bmatrix} -1.0 \\ 6.0 \end{bmatrix} (0.9) = \begin{bmatrix} -0.90 \\ 5.40 \end{bmatrix}$

Blending them and adding to Layer 2's frozen base output and residual stream $\tilde{h}^{(2)}$:
$$\text{LoRA}^{(2)}(\tilde{h}^{(2)}) = 0.7049 \begin{bmatrix} 0.60 \\ -0.60 \end{bmatrix} + 0.2951 \begin{bmatrix} -0.90 \\ 5.40 \end{bmatrix} = \begin{bmatrix} \mathbf{0.1574} \\ \mathbf{1.1706} \end{bmatrix}$$

$$h^{(2)} = \tilde{h}^{(2)} + \text{MoE}^{(2)}(\tilde{h}^{(2)}) + \text{LoRA}^{(2)}(\tilde{h}^{(2)}) = \begin{bmatrix} 3.0000 \\ 3.0000 \end{bmatrix} + \begin{bmatrix} 0.4220 \\ -1.1330 \end{bmatrix} + \begin{bmatrix} 0.1574 \\ 1.1706 \end{bmatrix} = \begin{bmatrix} \mathbf{3.5794} \\ \mathbf{3.0376} \end{bmatrix}$$

---

#### Summary Across Both Layers

| Layer | Token Entering MoE²-LoRA ($\tilde{h}^{(\ell)}$) | Global Pool Adapters Selected | Output Leaving Layer ($h^{(\ell)}$) |
| :--- | :--- | :--- | :--- |
| **Layer 1** | $[1.0000, \, 2.0000]$ | **Global Adapter 1** (63.4%) + **Global Adapter 2** (36.6%) | $[3.4101, \, 2.6913]$ |
| **Layer 2** | $[3.0000, \, 3.0000]$ | **Global Adapter 3** (70.5%) + **Global Adapter 2** (29.5%) | $[3.5794, \, 3.0376]$ |
